# Event Proximity Features

## Purpose

Build daily **event proximity features** from the M5 `calendar.csv`. The EDA (Section 7) showed that aggregate event-day sales are actually **lower** than normal days (-5.3%), driven by store closures on National holidays. The real demand signal is in **pre-event shopping behavior** — consumers stock up days before events, not on the day itself. Binary event flags miss this entirely.

We also assign **magnitude scores** (1–5) because not all events are equal. Christmas (5) drives weeks of elevated spending; Columbus Day (1) barely moves the needle.

In [1]:
# ── Uncomment below when running on Google Colab ────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.chdir('/content/drive/MyDrive/Colab Notebooks/m5-forecasting-accuracy/notebooks')

BASE_DIR = '..'
RAW_DIR = f'{BASE_DIR}/data/raw'
CLEAN_DIR = f'{BASE_DIR}/data/clean'
FIG_DIR = f'{BASE_DIR}/figures'

import os
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
print(f'Project dir: {BASE_DIR}')

Project dir: ..


## Step 0: Setup

In [2]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# RAW_DIR defined in mount cell above
# CLEAN_DIR defined in mount cell above

DATE_START = '2011-01-29'
DATE_END = '2016-06-19'

print(f"Date range: {DATE_START} to {DATE_END}")

Date range: 2011-01-29 to 2016-06-19


## Step 1: Extract all events from calendar.csv

The M5 calendar has two event slots per day: `event_name_1`/`event_type_1` and `event_name_2`/`event_type_2`. Most days use only slot 1, but a few days have two overlapping events (e.g., Easter and Pesach End on the same day).

We extract both slots, concatenate them into a single long-format table, and sort by date. The result: one row per `(event_name, date)` pair — the complete set of M5 events.

In [3]:
calendar = pd.read_csv(f'{RAW_DIR}/calendar.csv', parse_dates=['date'])

# Extract events from both slots
e1 = calendar[calendar['event_name_1'].notna()][['date', 'event_name_1', 'event_type_1']].copy()
e1 = e1.rename(columns={'event_name_1': 'event_name', 'event_type_1': 'event_type'})

e2 = calendar[calendar['event_name_2'].notna()][['date', 'event_name_2', 'event_type_2']].copy()
e2 = e2.rename(columns={'event_name_2': 'event_name', 'event_type_2': 'event_type'})

all_events = pd.concat([e1, e2], ignore_index=True).sort_values('date').reset_index(drop=True)

print(f"Total event entries: {len(all_events)}")
print(f"Unique events: {all_events['event_name'].nunique()}")
print(f"Days with 2 events: {calendar['event_name_2'].notna().sum()}")
print(f"\nEvent types: {dict(all_events['event_type'].value_counts())}")
print(f"\nAll {all_events['event_name'].nunique()} events:")
for name in sorted(all_events['event_name'].unique()):
    etype = all_events[all_events['event_name'] == name]['event_type'].iloc[0]
    count = (all_events['event_name'] == name).sum()
    print(f"  [{etype:10s}] {name} ({count}x)")

Total event entries: 167
Unique events: 30
Days with 2 events: 5

Event types: {'Religious': np.int64(56), 'National': np.int64(52), 'Cultural': np.int64(41), 'Sporting': np.int64(18)}

All 30 events:
  [Religious ] Chanukah End (5x)
  [National  ] Christmas (5x)
  [Cultural  ] Cinco De Mayo (6x)
  [National  ] ColumbusDay (5x)
  [Cultural  ] Easter (6x)
  [Religious ] Eid al-Fitr (5x)
  [Religious ] EidAlAdha (5x)
  [Cultural  ] Father's day (6x)
  [Cultural  ] Halloween (5x)
  [National  ] IndependenceDay (5x)
  [National  ] LaborDay (5x)
  [Religious ] LentStart (6x)
  [Religious ] LentWeek2 (6x)
  [National  ] MartinLutherKingDay (5x)
  [National  ] MemorialDay (6x)
  [Cultural  ] Mother's day (6x)
  [Sporting  ] NBAFinalsEnd (6x)
  [Sporting  ] NBAFinalsStart (6x)
  [National  ] NewYear (5x)
  [Religious ] OrthodoxChristmas (5x)
  [Religious ] OrthodoxEaster (6x)
  [Religious ] Pesach End (6x)
  [National  ] PresidentsDay (6x)
  [Religious ] Purim End (6x)
  [Religious ] Ramadan s

## Step 2: Assign magnitude scores (1–5) by expected retail impact

Not all events drive equal retail demand. We assign a 1–5 magnitude score based on expected Walmart sales impact:

| Mag | Meaning | Events |
|---|---|---|
| **5** | Massive | Christmas, Thanksgiving, SuperBowl |
| **4** | Major — long-weekend surges | MemorialDay, IndependenceDay, LaborDay, NewYear |
| **3** | Moderate — category-specific | Easter, Halloween, Mother's/Father's day, NBAFinals, StPatricksDay |
| **2** | Mild — niche bumps | ValentinesDay, Cinco De Mayo, PresidentsDay, religious observances |
| **1** | Minimal | ColumbusDay, MartinLutherKingDay, VeteransDay, minor holidays |

**These are judgment calls, not data-derived values.** The model will learn its own weighting of this feature — our magnitude provides a reasonable prior signal that the model can amplify or attenuate.

In [4]:
MAGNITUDE = {
    # 5 — Massive
    'Christmas': 5, 'Thanksgiving': 5, 'SuperBowl': 5,
    # 4 — Major
    'MemorialDay': 4, 'IndependenceDay': 4, 'LaborDay': 4, 'NewYear': 4,
    # 3 — Moderate
    'Easter': 3, 'Halloween': 3, "Mother's day": 3, "Father's day": 3,
    'NBAFinalsStart': 3, 'NBAFinalsEnd': 3, 'StPatricksDay': 3,
    # 2 — Mild
    'ValentinesDay': 2, 'Cinco De Mayo': 2, 'PresidentsDay': 2,
    'Ramadan starts': 2, 'Eid al-Fitr': 2, 'EidAlAdha': 2,
    'LentStart': 2, 'LentWeek2': 2, 'Pesach End': 2, 'OrthodoxEaster': 2,
    # 1 — Minimal
    'ColumbusDay': 1, 'MartinLutherKingDay': 1, 'VeteransDay': 1,
    'OrthodoxChristmas': 1, 'Purim End': 1, 'Chanukah End': 1,
}

all_events['magnitude'] = all_events['event_name'].map(MAGNITUDE)

unmapped = all_events[all_events['magnitude'].isna()]['event_name'].unique()
assert len(unmapped) == 0, (
    f"Unmapped events found in M5 calendar: {list(unmapped)}. "
    f"Add them to the MAGNITUDE dict before proceeding — silent NaN propagation "
    f"would break downstream merges and model training."
)
print(f"All {all_events['event_name'].nunique()} events mapped to magnitude scores.")

print(f"\nMagnitude distribution:")
for mag in sorted(all_events['magnitude'].unique()):
    names = sorted(all_events[all_events['magnitude'] == mag]['event_name'].unique())
    print(f"  [{int(mag)}] {', '.join(names)}")

All 30 events mapped to magnitude scores.

Magnitude distribution:
  [1] Chanukah End, ColumbusDay, MartinLutherKingDay, OrthodoxChristmas, Purim End, VeteransDay
  [2] Cinco De Mayo, Eid al-Fitr, EidAlAdha, LentStart, LentWeek2, OrthodoxEaster, Pesach End, PresidentsDay, Ramadan starts, ValentinesDay
  [3] Easter, Father's day, Halloween, Mother's day, NBAFinalsEnd, NBAFinalsStart, StPatricksDay
  [4] IndependenceDay, LaborDay, MemorialDay, NewYear
  [5] Christmas, SuperBowl, Thanksgiving


## Step 3: Build the event dates table

We finalize the event dates table with year extracted for reference. This becomes `event_dates.csv` — one row per `(event, date)` occurrence.

In [5]:
event_dates = all_events.copy()
event_dates['year'] = event_dates['date'].dt.year
event_dates = event_dates.rename(columns={'date': 'event_date'})
event_dates = event_dates[['event_name', 'event_type', 'magnitude', 'year', 'event_date']]
event_dates = event_dates.sort_values('event_date').reset_index(drop=True)

print(f"Event dates: {len(event_dates)} rows, {event_dates['event_name'].nunique()} unique events")
print(f"Date range: {event_dates['event_date'].min().date()} to {event_dates['event_date'].max().date()}")
print()
event_dates.head(15)

Event dates: 167 rows, 30 unique events
Date range: 2011-02-06 to 2016-06-19



,event_name,event_type,magnitude,year,event_date
0,SuperBowl,Sporting,5,2011,2011-02-06
1,ValentinesDay,Cultural,2,2011,2011-02-14
2,PresidentsDay,National,2,2011,2011-02-21
3,LentStart,Religious,2,2011,2011-03-09
4,LentWeek2,Religious,2,2011,2011-03-16
5,StPatricksDay,Cultural,3,2011,2011-03-17
6,Purim End,Religious,1,2011,2011-03-20
7,Easter,Cultural,3,2011,2011-04-24
8,OrthodoxEaster,Religious,2,2011,2011-04-24
9,Pesach End,Religious,2,2011,2011-04-26


## Step 4: Compute event proximity features

For every date in the M5 range (1,969 days), we find the nearest event (past or future) within a 14-day window.

**Why cap at 14 days?** Beyond 2 weeks, people aren't shopping for or thinking about an event.

**How the lookup works**: `events_sorted` is pre-sorted by date, so the nearest future event is `iloc[0]` of the filtered future subset, and the nearest past event is `iloc[-1]` of the filtered past subset.

In [6]:
PROXIMITY_WINDOW = 14

def compute_event_proximity(event_dates_df, start_date, end_date, window=14):
    date_range = pd.date_range(start=start_date, end=end_date, freq='D')
    events_sorted = event_dates_df.sort_values('event_date').reset_index(drop=True)

    records = []
    for current_date in date_range:
        future_events = events_sorted[events_sorted['event_date'] >= current_date]
        past_events = events_sorted[events_sorted['event_date'] <= current_date]

        days_until = window
        days_since = window
        nearest_name = 'none'
        nearest_type = 'none'
        magnitude = 0

        if len(future_events) > 0:
            nearest_future = future_events.iloc[0]
            days_until = min((nearest_future['event_date'] - current_date).days, window)

        if len(past_events) > 0:
            nearest_past = past_events.iloc[-1]
            days_since = min((current_date - nearest_past['event_date']).days, window)

        if len(future_events) > 0 and days_until <= days_since:
            nearest_name = nearest_future['event_name']
            nearest_type = nearest_future['event_type']
            magnitude = nearest_future['magnitude'] if days_until <= window else 0
        elif len(past_events) > 0:
            nearest_name = nearest_past['event_name']
            nearest_type = nearest_past['event_type']
            magnitude = nearest_past['magnitude'] if days_since <= window else 0

        records.append({
            'date': current_date,
            'days_until_event': days_until,
            'days_since_event': days_since,
            'event_magnitude': magnitude,
            'nearest_event_name': nearest_name,
            'event_type': nearest_type,
        })

    return pd.DataFrame(records)

event_features = compute_event_proximity(event_dates, DATE_START, DATE_END, PROXIMITY_WINDOW)

print(f"Event features shape: {event_features.shape}")
print(f"\nMagnitude distribution:")
print(event_features['event_magnitude'].value_counts().sort_index())
print(f"\nDays within 4 days of an event: "
      f"{(event_features['days_until_event'] <= 4).sum()} / {len(event_features)} "
      f"({100*(event_features['days_until_event'] <= 4).mean():.1f}%)")
print(f"\nNearest event distribution (top 10):")
print(event_features['nearest_event_name'].value_counts().head(10))

Event features shape: (1969, 6)

Magnitude distribution:
event_magnitude
1    415
2    573
3    444
4    321
5    216
Name: count, dtype: int64

Days within 4 days of an event: 716 / 1969 (36.4%)

Nearest event distribution (top 10):
nearest_event_name
LaborDay           143
ColumbusDay        123
Eid al-Fitr        119
Easter              92
Thanksgiving        87
Ramadan starts      87
StPatricksDay       83
MemorialDay         76
SuperBowl           75
IndependenceDay     73
Name: count, dtype: int64


## Step 5: Sanity checks

Spot-check specific dates to verify proximity logic:
- 2 days before Super Bowl should show SuperBowl approaching
- Thanksgiving day should show magnitude 5
- A mid-July date should show IndependenceDay recently passed

In [7]:
spot_checks = ['2016-02-05', '2015-11-26', '2015-12-25', '2014-07-06', '2013-05-10']
print("Spot checks:")
for d in spot_checks:
    row = event_features[event_features['date'] == d].iloc[0]
    print(f"  {d}: nearest={row['nearest_event_name']}, "
          f"days_until={row['days_until_event']}, "
          f"days_since={row['days_since_event']}, "
          f"magnitude={row['event_magnitude']}")

print(f"\nDates with no nearby event (magnitude=0): "
      f"{(event_features['event_magnitude'] == 0).sum()} / {len(event_features)} "
      f"({100*(event_features['event_magnitude'] == 0).mean():.1f}%)")

Spot checks:
  2016-02-05: nearest=SuperBowl, days_until=2, days_since=14, magnitude=5
  2015-11-26: nearest=Thanksgiving, days_until=0, days_since=0, magnitude=5
  2015-12-25: nearest=Christmas, days_until=0, days_since=0, magnitude=5
  2014-07-06: nearest=IndependenceDay, days_until=14, days_since=2, magnitude=4
  2013-05-10: nearest=Mother's day, days_until=2, days_since=5, magnitude=3

Dates with no nearby event (magnitude=0): 0 / 1969 (0.0%)


## Step 6: Save outputs

Two files:
1. **`event_dates.csv`** — All 30 M5 events with dates and magnitude scores
2. **`event_features.csv`** — Daily proximity features: one row per date

**Merge key for downstream**: `(date)` — events are national, same across all states.

In [8]:
event_dates.to_csv(f'{CLEAN_DIR}/event_dates.csv', index=False)
print(f"Saved: event_dates.csv ({len(event_dates)} rows, {event_dates['event_name'].nunique()} unique events)")

event_features.to_csv(f'{CLEAN_DIR}/event_features.csv', index=False)
print(f"Saved: event_features.csv ({len(event_features)} rows)")

print(f"\nColumns in event_features: {list(event_features.columns)}")
print(f"Merge key for downstream: (date)")

Saved: event_dates.csv (167 rows, 30 unique events)
Saved: event_features.csv (1969 rows)

Columns in event_features: ['date', 'days_until_event', 'days_since_event', 'event_magnitude', 'nearest_event_name', 'event_type']
Merge key for downstream: (date)


---
## Conclusion

This notebook produces `event_features.csv` (daily proximity to nearest event, with magnitude) and `event_dates.csv` (the 30 unique M5 events with their assigned magnitude scores). All 30 events are mapped — the hard assertion in Step 2 ensures any future M5 update with a new event halts the pipeline rather than silently propagating NaN.

**Two design decisions worth noting**:
- **Magnitude scores are judgment calls (1–5)**, not data-derived. They provide a reasonable prior signal that the model can amplify or attenuate. An ablation with equal weighting would test sensitivity to these priors.
- **14-day proximity window** matches the typical pre-event shopping ramp; longer windows pick up unrelated demand noise.

**Merge key for downstream**: `(date)` — events are national, identical across states.